# Caco2_Wang ML Prediction Notebook

**Contents**
1. Introduction
2. Background: Caco-2 permeability and importance
3. Dataset: TDC Caco2_Wang — retrieval and preprocessing
4. Methods: features, models, validation (reproducible pipeline)
5. Results: predictions, metrics, interpretation
6. Example use-cases
7. Limitations and next steps
8. Conclusion
9. Citations


## Introduction
This notebook reproduces the Caco2_Wang ML report: it retrieves the TDC Caco2_Wang dataset,
featurizes molecules using RDKit, trains multiple models (Random Forest, XGBoost/LightGBM, and an optional GNN),
evaluates performance on a held-out test set, and saves predictions and trained models.

Notes:
- The GNN section is optional and disabled by default (set USE_GNN=True to enable).
- This notebook may take minutes to hours depending on chosen models and hardware.


## Install / Verify Packages (optional)
The following cell contains a guarded pip install. Uncomment and run if you need to install packages in the notebook environment.
For RDKit and PyTorch Geometric, conda is strongly recommended. See README.md for instructions.


In [ ]:
# Example guarded install (uncomment to run)
# import sys
# !{sys.executable} -m pip install tdc rdkit-pypi pandas numpy scikit-learn xgboost lightgbm joblib shap matplotlib seaborn torch torchvision torchaudio
# Note: installing rdkit-pypi and torch-geometric via pip may be challenging inside some environments.


## 1) Load TDC Caco2_Wang Dataset
The dataset is retrieved via the TDC ADME API. The returned dataframe typically contains 'smiles' and target column 'Y'.


In [ ]:
from tdc import ADME
import pandas as pd

data = ADME(name='Caco2_Wang')
df = data.get_dataframe()
print('Columns:', df.columns.tolist())
print('Shape:', df.shape)
df.head()


## 2) Clean and standardize SMILES
We parse SMILES to RDKit Mol objects, remove salts and invalid molecules.


In [ ]:
from rdkit import Chem
from rdkit.Chem import SaltRemover
from rdkit.Chem import rdmolfiles

remover = SaltRemover.SaltRemover()

def clean_smiles(smi):
    try:
        s = remover.StripMol(Chem.MolFromSmiles(smi), dontRemoveEverything=True) if smi is not None else None
        if s is None:
            return None
        Chem.SanitizeMol(s)
        return Chem.MolToSmiles(s, isomericSmiles=True)
    except Exception as e:
        return None

df['smiles_clean'] = df['smiles'].apply(clean_smiles)
df = df[df['smiles_clean'].notnull()].reset_index(drop=True)
df['mol'] = df['smiles_clean'].apply(lambda s: Chem.MolFromSmiles(s))
print('After cleaning:', df.shape)
df.head()


## 3) Featurization: RDKit descriptors and Morgan fingerprints
We compute a small set of RDKit descriptors and 2048-bit Morgan fingerprints.


In [ ]:
from rdkit.Chem import Descriptors, AllChem
import numpy as np

desc_funcs = [
    ('MolWt', Descriptors.MolWt),
    ('LogP', Descriptors.MolLogP),
    ('TPSA', Descriptors.TPSA),
    ('HDonors', Descriptors.NumHDonors),
    ('HAcceptors', Descriptors.NumHAcceptors),
    ('RotBonds', Descriptors.NumRotatableBonds)
]

for name, fn in desc_funcs:
    df[name] = df['mol'].apply(fn)

def morgan_fp(mol, radius=2, nBits=2048):
    arr = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nBits)
    return np.array(arr)

fps = np.vstack(df['mol'].apply(lambda m: morgan_fp(m)).values)
fp_cols = [f'FP_{i}' for i in range(fps.shape[1])]
fps_df = pd.DataFrame(fps, columns=fp_cols)
df = pd.concat([df, fps_df], axis=1)

print('Features shape:', df.shape)
df.head()


## 4) Prepare data matrices and train/test split
We define X and y, perform an 80/20 split, and store indices for reproducibility.


In [ ]:
target_col = 'Y'  # adjust if different
df = df[df[target_col].notnull()].reset_index(drop=True)
y = df[target_col].astype(float).values
exclude = ['smiles', 'smiles_clean', 'mol', target_col]
feature_cols = [c for c in df.columns if c not in exclude]
X = df[feature_cols].values

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index, test_size=0.2, random_state=42)

print('Train size:', X_train.shape[0], 'Test size:', X_test.shape[0])


## 5) Model 1: Random Forest baseline
Train with GridSearchCV and evaluate on the held-out test set.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

rf = RandomForestRegressor(random_state=42, n_jobs=-1)
param_grid = {'n_estimators':[200,500], 'max_depth':[10,20,None], 'min_samples_leaf':[1,2,4]}
gs = GridSearchCV(rf, param_grid, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
gs.fit(X_train, y_train)
best_rf = gs.best_estimator_
print('Best RF params:', gs.best_params_)

y_pred_rf = best_rf.predict(X_test)
rmse_rf = mean_squared_error(y_test, y_pred_rf, squared=False)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)
print('RF RMSE', rmse_rf, 'MAE', mae_rf, 'R2', r2_rf)

# Save model and predictions
joblib.dump(best_rf, 'saved_models/rf_model.joblib')
import pandas as pd
out_rf = pd.DataFrame({'index': idx_test, 'smiles': df.loc[idx_test,'smiles_clean'].values, 'y_true': y_test, 'y_pred_rf': y_pred_rf})
out_rf.to_csv('caco2_predictions_rf.csv', index=False)


## 6) Model 2: XGBoost / LightGBM baseline
We try XGBoost if available, otherwise LightGBM. Grid search for hyperparameters.


In [ ]:
try:
    import xgboost as xgb
    use_xgb = True
except Exception as e:
    use_xgb = False

if use_xgb:
    from xgboost import XGBRegressor
    model = XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1)
    param_grid = {'n_estimators':[200,500], 'max_depth':[3,6,10], 'learning_rate':[0.01,0.1]}
else:
    from lightgbm import LGBMRegressor
    model = LGBMRegressor(random_state=42, n_jobs=-1)
    param_grid = {'n_estimators':[200,500], 'max_depth':[-1,10,20], 'learning_rate':[0.01,0.1]}

gs2 = GridSearchCV(model, param_grid, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
gs2.fit(X_train, y_train)
best_gb = gs2.best_estimator_
print('Best GB params:', gs2.best_params_)

y_pred_gb = best_gb.predict(X_test)
rmse_gb = mean_squared_error(y_test, y_pred_gb, squared=False)
mae_gb = mean_absolute_error(y_test, y_pred_gb)
r2_gb = r2_score(y_test, y_pred_gb)
print('GB RMSE', rmse_gb, 'MAE', mae_gb, 'R2', r2_gb)

joblib.dump(best_gb, 'saved_models/gb_model.joblib')
out_gb = pd.DataFrame({'index': idx_test, 'smiles': df.loc[idx_test,'smiles_clean'].values, 'y_true': y_test, 'y_pred_gb': y_pred_gb})
out_gb.to_csv('caco2_predictions_gb.csv', index=False)


## 7) Optional Model 3: Graph Neural Network (GNN) baseline
Set USE_GNN = True if you have a GPU and PyTorch Geometric installed.


In [ ]:
USE_GNN = False  # change to True to enable GNN training
if USE_GNN:
    import torch
    from torch_geometric.data import Data, DataLoader
    from torch_geometric.nn import GCNConv, global_mean_pool
    # Convert RDKit mol to PyG Data object
    def mol_to_graph(mol):
        # Node features: atom one-hot (simple) + degree
        atom_feats = []
        edge_index = []
        for atom in mol.GetAtoms():
            atom_feats.append([atom.GetAtomicNum(), atom.GetDegree()])
        for bond in mol.GetBonds():
            a = bond.GetBeginAtomIdx()
            b = bond.GetEndAtomIdx()
            edge_index.append([a,b])
            edge_index.append([b,a])
        import torch
        x = torch.tensor(atom_feats, dtype=torch.float)
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        data = Data(x=x, edge_index=edge_index)
        return data

    # Prepare dataset
    graphs = [mol_to_graph(m) for m in df['mol']]
    targets = torch.tensor(df[target_col].values, dtype=torch.float)
    for i,g in enumerate(graphs):
        g.y = targets[i].unsqueeze(0)

    # Split indices
    train_idx = idx_train.tolist()
    test_idx = idx_test.tolist()
    train_graphs = [graphs[i] for i in train_idx]
    test_graphs = [graphs[i] for i in test_idx]

    train_loader = DataLoader(train_graphs, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_graphs, batch_size=32)

    # Simple GCN regressor
    import torch.nn as nn
    class GCNRegressor(nn.Module):
        def __init__(self, in_channels, hidden=64):
            super().__init__()
            self.conv1 = GCNConv(in_channels, hidden)
            self.conv2 = GCNConv(hidden, hidden)
            self.lin = nn.Linear(hidden, 1)
        def forward(self, x, edge_index, batch):
            x = self.conv1(x, edge_index).relu()
            x = self.conv2(x, edge_index).relu()
            x = global_mean_pool(x, batch)
            return self.lin(x).squeeze(1)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = GCNRegressor(in_channels=graphs[0].num_node_features).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    # Train loop (simple)
    for epoch in range(1, 21):
        model.train()
        total_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            pred = model(batch.x, batch.edge_index, batch.batch)
            loss = loss_fn(pred, batch.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * batch.num_graphs
        print(f'Epoch {epoch} train loss', total_loss / len(train_graphs))

    # Evaluate
    model.eval()
    preds = []
    trues = []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            pred = model(batch.x, batch.edge_index, batch.batch)
            preds.append(pred.cpu().numpy())
            trues.append(batch.y.cpu().numpy())
    import numpy as _np
    y_pred_gnn = _np.concatenate(preds)
    y_test_gnn = _np.concatenate(trues)
    rmse_gnn = mean_squared_error(y_test_gnn, y_pred_gnn, squared=False)
    print('GNN RMSE', rmse_gnn)
    torch.save(model.state_dict(), 'saved_models/gnn_model.pt')
    out_gnn = pd.DataFrame({'index': test_idx, 'smiles': df.loc[test_idx,'smiles_clean'].values, 'y_true': y_test_gnn, 'y_pred_gnn': y_pred_gnn})
    out_gnn.to_csv('caco2_predictions_gnn.csv', index=False)
else:
    print('GNN disabled. Set USE_GNN = True to enable (requires PyTorch Geometric).')


## 8) Applicability domain (simple nearest Tanimoto similarity)
Compute Tanimoto similarity of each test compound to nearest training compound using Morgan fingerprints.


In [ ]:
from rdkit.DataStructs.cDataStructs import TanimotoSimilarity
from rdkit.DataStructs import ExplicitBitVect
# Re-generate RDKit fingerprint objects for similarity calculation
fps_rdkit = [AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048) for m in df['mol']]
train_fps = [fps_rdkit[i] for i in idx_train]
test_fps = [fps_rdkit[i] for i in idx_test]
nearest_sims = []
for tf in test_fps:
    sims = [TanimotoSimilarity(tf, tr) for tr in train_fps]
    nearest_sims.append(max(sims))

ad_df = pd.DataFrame({'index': idx_test, 'nearest_tanimoto': nearest_sims})
ad_df.to_csv('caco2_applicability.csv', index=False)
ad_df.head()


## 9) Feature importance / SHAP (optional)
If SHAP is installed, compute SHAP values for the Random Forest or GB model to interpret feature contributions.


In [ ]:
try:
    import shap
    shap_installed = True
except Exception:
    shap_installed = False

if shap_installed:
    explainer = shap.Explainer(best_rf.predict, X_train)
    shap_values = explainer(X_test)
    shap.plots.beeswarm(shap_values)
else:
    print('SHAP not installed; skip feature importance.')


## 10) Save combined predictions and summary
Combine per-model predictions into one CSV and save summary metrics.


In [ ]:
# Load per-model predictions if exist and merge
preds = pd.DataFrame({'index': idx_test, 'smiles': df.loc[idx_test,'smiles_clean'].values, 'y_true': y_test})
try:
    rf_df = pd.read_csv('caco2_predictions_rf.csv')
    preds = preds.merge(rf_df[['index','y_pred_rf']], on='index')
except Exception:
    pass
try:
    gb_df = pd.read_csv('caco2_predictions_gb.csv')
    preds = preds.merge(gb_df[['index','y_pred_gb']], on='index')
except Exception:
    pass
try:
    gnn_df = pd.read_csv('caco2_predictions_gnn.csv')
    preds = preds.merge(gnn_df[['index','y_pred_gnn']], on='index')
except Exception:
    pass
preds.to_csv('caco2_predictions.csv', index=False)
# Summary metrics
summary = {'model': ['RandomForest','GB','GNN'], 'RMSE':[rmse_rf if 'rmse_rf' in globals() else None, rmse_gb if 'rmse_gb' in globals() else None, rmse_gnn if 'rmse_gnn' in globals() else None], 'MAE':[mae_rf if 'mae_rf' in globals() else None, mae_gb if 'mae_gb' in globals() else None, None], 'R2':[r2_rf if 'r2_rf' in globals() else None, r2_gb if 'r2_gb' in globals() else None, None]}
summary_df = pd.DataFrame(summary)
summary_df.to_csv('caco2_summary_metrics.csv', index=False)
summary_df


## 11) Conclusion and Next Steps
- The notebook provides a reproducible pipeline to train baseline models for Caco-2 permeability.
- Next steps: advanced GNN architectures, transfer learning, nested CV, and more careful curation.
